# 09 Export Final Model for Streamlit App

This notebook trains the final Gradient Boosting model using the extracted physiological features and saves the model artifacts required by the Streamlit application.

In [1]:
# Libraries for file paths, JSON metadata, and model saving
from pathlib import Path
import pickle
import json

import pandas as pd

# Scikit-learn pipeline and final model
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier

In [2]:
# Define project folders
PROJECT_ROOT = Path("..")

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
APP_ARTIFACTS_DIR = PROJECT_ROOT / "app_artifacts"

# Create app artifact folder if it does not exist
APP_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Feature dataset path
features_path = PROCESSED_DATA_DIR / "features.csv"

print("Feature file exists:", features_path.exists())
print("App artifacts folder:", APP_ARTIFACTS_DIR)

Feature file exists: True
App artifacts folder: ..\app_artifacts


In [3]:
# Load extracted feature dataset from Notebook 03
features_df = pd.read_csv(features_path)

print("Feature dataset shape:", features_df.shape)

features_df.head()

Feature dataset shape: (2151, 75)


,subject_id,window_id,start_sec,end_sec,majority_ratio,ECG_mean,ECG_std,ECG_min,ECG_max,ECG_median,...,Temp_rms,Temp_energy,Temp_mean_abs_change,Temp_std_change,Temp_max_abs_change,Temp_zero_crossings,Temp_skewness,Temp_kurtosis,label,label_name
0,S10,0,90.0,120.0,0.904762,0.001516,0.132416,-0.663895,0.785843,0.030716,...,33.897731,1149.056162,0.022942,0.032233,0.183929,0,-0.302280,3.753279,1,Neutral / Baseline
1,S10,1,105.0,135.0,1.000000,0.000542,0.136359,-0.574814,0.821457,0.031586,...,33.922503,1150.736193,0.024211,0.033127,0.177979,0,-0.395398,4.197550,1,Neutral / Baseline
2,S10,2,120.0,150.0,1.000000,0.001889,0.140376,-0.667923,0.821457,0.028976,...,33.949085,1152.540389,0.024425,0.033431,0.177979,0,-0.298644,3.715316,1,Neutral / Baseline
3,S10,3,135.0,165.0,1.000000,0.001967,0.154921,-0.667923,0.838211,0.019730,...,33.982891,1154.836868,0.024582,0.033548,0.179626,0,-0.029038,3.213001,1,Neutral / Baseline
4,S10,4,150.0,180.0,1.000000,0.000855,0.161409,-0.664948,0.838211,0.005791,...,34.028476,1157.937208,0.026112,0.035063,0.179871,0,-0.213312,2.793012,1,Neutral / Baseline


In [4]:
# Columns that are not model input features
metadata_columns = [
    "subject_id",
    "window_id",
    "start_sec",
    "end_sec",
    "majority_ratio",
    "label",
    "label_name"
]

# Select only physiological feature columns
feature_columns = [
    column for column in features_df.columns
    if column not in metadata_columns
]

# Input features and target labels
X = features_df[feature_columns].copy()
y = features_df["label"].copy()

# Fill missing values if any exist
X = X.fillna(X.mean())

print("Number of samples:", X.shape[0])
print("Number of features:", X.shape[1])
print("Labels:", sorted(y.unique()))

Number of samples: 2151
Number of features: 68
Labels: [np.int64(1), np.int64(2), np.int64(3)]


In [5]:
# Create final model pipeline
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", GradientBoostingClassifier(
        random_state=42
    ))
])

# Train final model on all available feature data
final_model.fit(X, y)

print("Final Gradient Boosting model trained.")

Final Gradient Boosting model trained.


In [6]:
# Save trained model pipeline
model_path = APP_ARTIFACTS_DIR / "stress_model.pkl"

with open(model_path, "wb") as file:
    pickle.dump(final_model, file)

print("Saved model to:", model_path)

Saved model to: ..\app_artifacts\stress_model.pkl


In [7]:
# Save feature column order required by the model
feature_columns_path = APP_ARTIFACTS_DIR / "feature_columns.json"

with open(feature_columns_path, "w") as file:
    json.dump(feature_columns, file, indent=4)

print("Saved feature columns to:", feature_columns_path)

Saved feature columns to: ..\app_artifacts\feature_columns.json


In [8]:
# Save model information for the Streamlit app
model_metadata = {
    "model_name": "Gradient Boosting",
    "project_name": "Wearable Stress Detection",
    "dataset": "WESAD",
    "signals_used": [
        "ECG",
        "EDA/GSR",
        "Respiration",
        "Temperature"
    ],
    "classes": {
        "1": "Neutral / Baseline",
        "2": "Stress",
        "3": "Amusement"
    },
    "number_of_features": len(feature_columns),
    "number_of_windows": len(features_df),
    "evaluation": {
        "loso_mean_accuracy": 0.6871,
        "loso_std_accuracy": 0.1638,
        "loso_mean_macro_f1": 0.5769,
        "loso_std_macro_f1": 0.1805
    }
}

metadata_path = APP_ARTIFACTS_DIR / "model_metadata.json"

with open(metadata_path, "w") as file:
    json.dump(model_metadata, file, indent=4)

print("Saved model metadata to:", metadata_path)

Saved model metadata to: ..\app_artifacts\model_metadata.json


In [9]:
# Create an empty feature template for users
template_df = pd.DataFrame(columns=feature_columns)

template_path = APP_ARTIFACTS_DIR / "feature_input_template.csv"

template_df.to_csv(template_path, index=False)

print("Saved input template to:", template_path)

Saved input template to: ..\app_artifacts\feature_input_template.csv


In [10]:
import joblib

model_path = APP_ARTIFACTS_DIR / "stress_model.joblib"

joblib.dump(final_model, model_path)

print("Saved model to:", model_path)

Saved model to: ..\app_artifacts\stress_model.joblib
